# An HDG method for the Laplace-Beltrami problem on surfaces

We propose and analyze an HDG method for the Laplace-Beltrami problem:
$$
 \Delta_{\Gamma} u = f \quad \text{on } \Gamma.
$$

## Define surface, exact solution, and rhs

In [1]:
from manufactured_solution_tools import get_solution_str, eprint

In [18]:
# Level set define Gamma
R = 1.0
phi_str = "x*x + y*y + z*z -" + str(R)

# Exact solution
u_str = "sin(x/2)*sin(y/2)*sin(z/2)"
levelset_str = "x**2 + y**2 + z**2 - 1"
f, tang_grad_u = get_solution_str(u_str, levelset_str)

eprint(f, name="f")
eprint(tang_grad_u, name="\\nabla_{\\Gamma}u")
f_str = str(f)

Got the level set function:  x**2 + y**2 + z**2 - 1
Normal: Matrix([[1.0*x/(x**2 + y**2 + z**2)**0.5], [1.0*y/(x**2 + y**2 + z**2)**0.5], [1.0*z/(x**2 + y**2 + z**2)**0.5]])
b:  Matrix([[0], [0], [0]])
c:  1
epsilon:  1
RHS f:  1.0*x*(-0.25*x*sin(x/2)*sin(y/2)*sin(z/2)/(x**2 + y**2 + z**2)**0.5 + 0.25*y*sin(z/2)*cos(x/2)*cos(y/2)/(x**2 + y**2 + z**2)**0.5 + 0.25*z*sin(y/2)*cos(x/2)*cos(z/2)/(x**2 + y**2 + z**2)**0.5)/(x**2 + y**2 + z**2)**0.5 + 1.0*y*(0.25*x*sin(z/2)*cos(x/2)*cos(y/2)/(x**2 + y**2 + z**2)**0.5 - 0.25*y*sin(x/2)*sin(y/2)*sin(z/2)/(x**2 + y**2 + z**2)**0.5 + 0.25*z*sin(x/2)*cos(y/2)*cos(z/2)/(x**2 + y**2 + z**2)**0.5)/(x**2 + y**2 + z**2)**0.5 + 1.0*z*(0.25*x*sin(y/2)*cos(x/2)*cos(z/2)/(x**2 + y**2 + z**2)**0.5 + 0.25*y*sin(x/2)*cos(y/2)*cos(z/2)/(x**2 + y**2 + z**2)**0.5 - 0.25*z*sin(x/2)*sin(y/2)*sin(z/2)/(x**2 + y**2 + z**2)**0.5)/(x**2 + y**2 + z**2)**0.5 + (0.5*x*sin(y/2)*sin(z/2)*cos(x/2)/(x**2 + y**2 + z**2)**0.5 + 0.5*y*sin(x/2)*sin(z/2)*cos(y/2)/(x**2 + y**2 + z

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [6]:
import netgen.gui
%gui tk
from math import pi
from ngsolve import *

Choose orders for space and geometry approximation

In [7]:
order = 2 # Order of finite elemet space
order_g = order # Geometry order

## 1. Generating the surface mesh

In [8]:
from netgen.csg import *
geo = CSGeometry()
geo.Add(Sphere(Pnt(0,0,0), 1).bc("sphere")) 
# Q: bc?

0

In [9]:
from netgen.meshing import MeshingParameters, MeshingStep
mp = MeshingParameters(maxh=0.5, perfstepsend=MeshingStep.MESHSURFACE)
# TODO: Find a list over all meshing parameters
mesh = Mesh(geo.GenerateMesh(mp=mp))

 Start Findpoints
 Analyze spec points
 Find edges
 Start Findpoints
 Analyze spec points
 Find edges
 Start Findpoints
 Analyze spec points
 Find edges
 Surface 1 / 1
 Optimize Surface


In [12]:
if order_g > 0:
    mesh.Curve(order_g)
Draw(mesh)

 Curve elements, order = 2


## Surface mesh related quantities

In [13]:
print("Mesh dim %d" % mesh.dim)
n_K = specialcf.normal(mesh.dim)
h = specialcf.mesh_size
# Should be a projector not a vector? Look up documentation!
t_e = specialcf.tangential(mesh.dim)
n_e = Cross(n_K, t_e)
#specialcf.tangential?

Mesh dim 3


## 2. Discretization

### Surface finite element spaces

In [14]:
Vh_s = SurfaceL2(mesh, order=order)
Vh_f = FacetSurface(mesh, order=order)

Vh = FESpace([Vh_s, Vh_f])
u, u_f = Vh.TrialFunction()
v, v_f = Vh.TestFunction() 

## Surface gradients, co-normal derivatives and jumps

In [15]:
grad_u = u.Trace().Deriv()
grad_v = v.Trace().Deriv()

du_ne = InnerProduct(grad_u, n_e)
dv_ne = InnerProduct(grad_v, n_e)

## Bilinear and linear forms

In [16]:
a = BilinearForm(Vh)
alpha = 10*(order+1)**2 # Shouldn't it be just order?

a += SymbolicBFI(grad_u * grad_v, VOL_or_BND = BND)
a += SymbolicBFI(u*v, VOL_or_BND = BND)
a += SymbolicBFI(-du_ne * (v - v_f), element_boundary=True, VOL_or_BND = BND)
a += SymbolicBFI(-dv_ne * (u - u_f), element_boundary=True, VOL_or_BND = BND)
a += SymbolicBFI(alpha/h*(u - u_f)*(v-v_f), element_boundary=True, VOL_or_BND = BND)

In [19]:
#f_str = "sin(x)*sin(y)*sin(z)"
print(f_str)
f = eval(f_str)
print(f)
do_compile = False
do_wait = True
f_form = (f*v).Compile(do_compile, wait=do_wait)
l = LinearForm(Vh)
l += SymbolicLFI(form = f*v, VOL_or_BND = BND)

1.0*x*(-0.25*x*sin(x/2)*sin(y/2)*sin(z/2)/(x**2 + y**2 + z**2)**0.5 + 0.25*y*sin(z/2)*cos(x/2)*cos(y/2)/(x**2 + y**2 + z**2)**0.5 + 0.25*z*sin(y/2)*cos(x/2)*cos(z/2)/(x**2 + y**2 + z**2)**0.5)/(x**2 + y**2 + z**2)**0.5 + 1.0*y*(0.25*x*sin(z/2)*cos(x/2)*cos(y/2)/(x**2 + y**2 + z**2)**0.5 - 0.25*y*sin(x/2)*sin(y/2)*sin(z/2)/(x**2 + y**2 + z**2)**0.5 + 0.25*z*sin(x/2)*cos(y/2)*cos(z/2)/(x**2 + y**2 + z**2)**0.5)/(x**2 + y**2 + z**2)**0.5 + 1.0*z*(0.25*x*sin(y/2)*cos(x/2)*cos(z/2)/(x**2 + y**2 + z**2)**0.5 + 0.25*y*sin(x/2)*cos(y/2)*cos(z/2)/(x**2 + y**2 + z**2)**0.5 - 0.25*z*sin(x/2)*sin(y/2)*sin(z/2)/(x**2 + y**2 + z**2)**0.5)/(x**2 + y**2 + z**2)**0.5 + (0.5*x*sin(y/2)*sin(z/2)*cos(x/2)/(x**2 + y**2 + z**2)**0.5 + 0.5*y*sin(x/2)*sin(z/2)*cos(y/2)/(x**2 + y**2 + z**2)**0.5 + 0.5*z*sin(x/2)*sin(y/2)*cos(z/2)/(x**2 + y**2 + z**2)**0.5)*(-1.0*x**2/(x**2 + y**2 + z**2)**1.5 - 1.0*y**2/(x**2 + y**2 + z**2)**1.5 - 1.0*z**2/(x**2 + y**2 + z**2)**1.5 + 3.0/(x**2 + y**2 + z**2)**0.5) + 7*sin(x/2)

## Assemble and solve

In [20]:
a.Assemble()
l.Assemble()

In [21]:
u_h = GridFunction(Vh)
u_s_h = u_h.components[0]
u_h.vec.data = a.mat.Inverse(freedofs=Vh.FreeDofs()) * l.vec
Draw(u_s_h, mesh, "u")

In [22]:
u_ex = eval(u_str)
err_sqr_coefs = (u_ex-u_s_h)**2
l2error = sqrt(Integrate(cf = err_sqr_coefs, mesh=mesh, order=order, VOL_or_BND = BND) )
print("||u-u_h||_L2 = %e" % l2error)

||u-u_h||_L2 = 3.318389e-04
